# Phase M5 — Video Data Preparation
## Extract 2D Axial Slices & Build Triplets for Diffusion Training

**Pipeline:** MU-Glioma (training) + BraTS-PTG (validation)  
**Output:** `/kaggle/working/video_data/*.npz` — ready for DDPM training

### Kaggle Datasets Attached:
1. **MU-Glioma-Post External Validation** — `MU-Glioma-Post/PatientID_XXXX/Timepoint_Y/*.nii_gz`
2. **BraTS 2024 Training Data** — `training_data/BraTS-GLI-XXXXX-YYY/*.nii_gz`
3. **BraTS 2024 Metadata** — `scan_index.json`, `tumor_volumes.csv`

---

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 1: Imports & Kaggle .nii.gz Symlink Fix
# ═══════════════════════════════════════════════════════════════
import os, json, time, warnings, gc
import numpy as np
import nibabel as nib
from pathlib import Path
from collections import defaultdict
from scipy.ndimage import zoom

warnings.filterwarnings('ignore')

SLICE_SIZE  = 128
OUTPUT_DIR  = Path('/kaggle/working/video_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SYMLINK_DIR = Path('/kaggle/working/nifti_links')

# ── Symlink .nii_gz → .nii.gz (Kaggle renames the extension) ──
def setup_nii_gz_symlinks(data_dir):
    count = 0
    for nii_gz in Path(data_dir).rglob('*.nii_gz'):
        rel = nii_gz.relative_to(data_dir)
        real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
        link = SYMLINK_DIR / rel.parent / real_name
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            os.symlink(str(nii_gz), str(link))
            count += 1
    return count

DATA_ROOT = Path('/kaggle/input')
total_links = 0
for ds_dir in DATA_ROOT.rglob('*'):
    if ds_dir.is_dir() and list(ds_dir.glob('*.nii_gz')):
        n = setup_nii_gz_symlinks(ds_dir.parent)
        total_links += n
        break
# Blanket scan all input
for ds_dir in DATA_ROOT.iterdir():
    if ds_dir.is_dir():
        n = setup_nii_gz_symlinks(ds_dir)
        if n: 
            print(f'  Symlinks: {n} from {ds_dir.name}')
            total_links += n

print(f'Total symlinks created: {total_links}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 2: Discover Exact Dataset Paths
# ═══════════════════════════════════════════════════════════════
# Find MU-Glioma-Post root (contains PatientID_XXXX folders)
MU_GLIOMA_ROOT = None
for p in SYMLINK_DIR.rglob('PatientID_0003'):
    MU_GLIOMA_ROOT = p.parent; break
if MU_GLIOMA_ROOT is None:
    for p in DATA_ROOT.rglob('PatientID_0003'):
        MU_GLIOMA_ROOT = p.parent; break

# Find BraTS-PTG root (contains BraTS-GLI-XXXXX-YYY folders)
BRATS_PTG_ROOT = None
for p in SYMLINK_DIR.rglob('BraTS-GLI-00005-100'):
    BRATS_PTG_ROOT = p.parent; break
if BRATS_PTG_ROOT is None:
    for p in DATA_ROOT.rglob('BraTS-GLI-00005-100'):
        BRATS_PTG_ROOT = p.parent; break

# Find BraTS metadata (scan_index.json for BraTS volumes)
BRATS_META_DIR = None
for p in DATA_ROOT.rglob('tumor_volumes.csv'):
    BRATS_META_DIR = p.parent; break

print(f'MU-Glioma root:  {MU_GLIOMA_ROOT}')
print(f'BraTS-PTG root:  {BRATS_PTG_ROOT}')
print(f'BraTS metadata:  {BRATS_META_DIR}')

if MU_GLIOMA_ROOT is None:
    raise RuntimeError('MU-Glioma-Post data not found! Check Kaggle input datasets.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 3: Core Functions — Slice Extraction
# ═══════════════════════════════════════════════════════════════
# KEY FIX: For each patient, we find the best z-slice at BASELINE
# and use that SAME z for ALL timepoints. This ensures temporal
# consistency — the video tracks the same anatomical plane.

def resize_slice(arr, target_size, order=0):
    h, w = arr.shape
    if h == target_size and w == target_size:
        return arr
    return zoom(arr, (target_size / h, target_size / w), order=order)

def find_best_z(seg_path):
    """Find axial slice with max tumor area (used for baseline only)."""
    seg_vol = nib.load(seg_path).get_fdata().astype(np.uint8)
    tumor_area_per_z = (seg_vol > 0).sum(axis=(0, 1))
    if tumor_area_per_z.max() == 0:
        return seg_vol.shape[2] // 2
    return int(tumor_area_per_z.argmax())

def extract_at_z(seg_path, t1c_path, z, size=128):
    """Extract axial slice at a FIXED z-index, resize to size x size."""
    seg_vol = nib.load(seg_path).get_fdata().astype(np.uint8)
    t1c_vol = nib.load(t1c_path).get_fdata().astype(np.float32)

    # Clamp z to valid range (tumor may shift across timepoints)
    z = min(max(z, 0), seg_vol.shape[2] - 1)

    seg_s = resize_slice(seg_vol[:, :, z], size, order=0).astype(np.uint8)
    t1c_s = resize_slice(t1c_vol[:, :, z], size, order=1).astype(np.float32)

    vmin, vmax = t1c_s.min(), t1c_s.max()
    t1c_s = (t1c_s - vmin) / (vmax - vmin + 1e-8) if vmax - vmin > 1e-6 else np.zeros_like(t1c_s)

    return seg_s, t1c_s, {
        'wt': int((seg_s > 0).sum()),
        'tc': int(np.isin(seg_s, [1, 4]).sum()),
        'et': int((seg_s == 4).sum()), 'z': z
    }

print('Core functions ready (fixed z-slice tracking).')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 4: Extract MU-Glioma Slices (Fixed Z-Tracking)
# ═══════════════════════════════════════════════════════════════
t0 = time.time()
mu_patients = defaultdict(list)
corrupted = []

patient_dirs = sorted([d for d in MU_GLIOMA_ROOT.iterdir()
                       if d.is_dir() and d.name.startswith('PatientID')])
print(f'Found {len(patient_dirs)} MU-Glioma patient directories')

for pi, pid_dir in enumerate(patient_dirs):
    pid = pid_dir.name
    tp_dirs = sorted([d for d in pid_dir.iterdir()
                      if d.is_dir() and d.name.startswith('Timepoint')])
    if len(tp_dirs) < 2:
        continue

    # Collect valid timepoints (skip corrupted files)
    valid_tps = []
    for tp_dir in tp_dirs:
        seg_files = list(tp_dir.glob('*tumorMask*.nii.gz'))
        t1c_files = list(tp_dir.glob('*t1c*.nii.gz'))
        if seg_files and t1c_files:
            valid_tps.append((tp_dir, str(seg_files[0]), str(t1c_files[0])))

    if len(valid_tps) < 2:
        continue

    # ── Step 1: Find best z at baseline — try each timepoint until one works ──
    baseline_z = None
    for _, seg_path, _ in valid_tps:
        try:
            baseline_z = find_best_z(seg_path)
            break
        except Exception as e:
            corrupted.append(seg_path)
            continue

    if baseline_z is None:
        corrupted.append(f'{pid}: ALL timepoints corrupted')
        continue

    # ── Step 2: Extract same z-slice for ALL timepoints ──
    for tp_dir, seg_path, t1c_path in valid_tps:
        try:
            seg_s, t1c_s, meta = extract_at_z(seg_path, t1c_path, baseline_z, SLICE_SIZE)
            tp_num = int(tp_dir.name.replace('Timepoint_', ''))
            mu_patients[pid].append({
                'seg': seg_s, 't1c': t1c_s, 'tp_num': tp_num, **meta
            })
        except Exception as e:
            corrupted.append(f'{pid}/{tp_dir.name}: {e}')

    if (pi + 1) % 50 == 0:
        print(f'  Processed {pi+1}/{len(patient_dirs)}...')

# Sort and filter
for pid in list(mu_patients.keys()):
    mu_patients[pid].sort(key=lambda x: x['tp_num'])
    if len(mu_patients[pid]) < 2:
        del mu_patients[pid]

elapsed = time.time() - t0
print(f'\nMU-Glioma done ({elapsed:.0f}s)')
print(f'  Patients (≥2 tp): {len(mu_patients)}')
print(f'  Total scans:      {sum(len(v) for v in mu_patients.values())}')
print(f'  Corrupted/skipped: {len(corrupted)}')
if corrupted:
    for c in corrupted[:10]:
        print(f'    ⚠ {c}')
tp_counts = [len(v) for v in mu_patients.values()]
for n in sorted(set(tp_counts)):
    print(f'    {n} timepoints: {tp_counts.count(n)} patients')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 5: Build MU-Glioma Training Triplets
# ═══════════════════════════════════════════════════════════════
seg_starts, seg_ends, seg_gts = [], [], []
t1c_starts, t_interps, pids_list, types_list = [], [], [], []
n_triplet = n_identity = 0

for pid, scans in mu_patients.items():
    S = scans
    if len(S) >= 3:
        for i in range(len(S)):
            for k in range(i+2, len(S)):
                for j in range(i+1, k):
                    t_val = (j - i) / (k - i)
                    seg_starts.append(S[i]['seg']); seg_ends.append(S[k]['seg'])
                    seg_gts.append(S[j]['seg']); t1c_starts.append(S[i]['t1c'])
                    t_interps.append(t_val); pids_list.append(pid)
                    types_list.append('triplet'); n_triplet += 1

    for i in range(len(S)-1):
        seg_starts.append(S[i]['seg']); seg_ends.append(S[i+1]['seg'])
        seg_gts.append(S[i]['seg']); t1c_starts.append(S[i]['t1c'])
        t_interps.append(0.0); pids_list.append(pid)
        types_list.append('identity_0'); n_identity += 1

        seg_starts.append(S[i]['seg']); seg_ends.append(S[i+1]['seg'])
        seg_gts.append(S[i+1]['seg']); t1c_starts.append(S[i]['t1c'])
        t_interps.append(1.0); pids_list.append(pid)
        types_list.append('identity_1'); n_identity += 1

print(f'Triplets: {n_triplet} real + {n_identity} identity = {n_triplet+n_identity} total')

save_path = OUTPUT_DIR / 'mu_glioma_triplets.npz'
np.savez_compressed(str(save_path),
    seg_start=np.array(seg_starts, dtype=np.uint8),
    seg_end=np.array(seg_ends, dtype=np.uint8),
    seg_gt=np.array(seg_gts, dtype=np.uint8),
    t1c_start=np.array(t1c_starts, dtype=np.float32),
    t_interp=np.array(t_interps, dtype=np.float32),
    pids=np.array(pids_list), types=np.array(types_list))
print(f'✅ Saved → {save_path}  ({save_path.stat().st_size/1e6:.1f} MB)')

del seg_starts, seg_ends, seg_gts, t1c_starts; gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 6: Extract BraTS-PTG Slices (Fixed Z-Tracking)
# ═══════════════════════════════════════════════════════════════
t0 = time.time()
brats_patients = defaultdict(list)
brats_corrupted = []

if BRATS_PTG_ROOT is None:
    print('⚠ BraTS-PTG not found — skipping.')
else:
    scan_dirs = sorted([d for d in BRATS_PTG_ROOT.iterdir()
                        if d.is_dir() and d.name.startswith('BraTS-GLI')])
    print(f'Found {len(scan_dirs)} BraTS-PTG scan directories')

    # Group by patient
    brats_scan_info = defaultdict(list)
    for scan_dir in scan_dirs:
        scan_name = scan_dir.name
        parts = scan_name.rsplit('-', 1)
        if len(parts) != 2: continue
        pid = parts[0]
        try: tp_num = int(parts[1])
        except ValueError: continue
        seg_path = scan_dir / f'{scan_name}-seg.nii.gz'
        t1c_path = scan_dir / f'{scan_name}-t1c.nii.gz'
        if seg_path.exists() and t1c_path.exists():
            brats_scan_info[pid].append((tp_num, str(seg_path), str(t1c_path), scan_name))

    for pid, scans in brats_scan_info.items():
        scans.sort(key=lambda x: x[0])

        # Find baseline z (try each until one works)
        baseline_z = None
        for _, seg_path, _, sname in scans:
            try:
                baseline_z = find_best_z(seg_path)
                break
            except Exception:
                brats_corrupted.append(sname)
                continue

        if baseline_z is None: continue

        for tp_num, seg_path, t1c_path, scan_name in scans:
            try:
                seg_s, t1c_s, meta = extract_at_z(seg_path, t1c_path, baseline_z, SLICE_SIZE)
                brats_patients[pid].append({
                    'seg': seg_s, 't1c': t1c_s, 'tp_num': tp_num,
                    'name': scan_name, **meta
                })
            except Exception as e:
                brats_corrupted.append(f'{scan_name}: {e}')

    for pid in brats_patients:
        brats_patients[pid].sort(key=lambda x: x['tp_num'])

    eval_pats = {p: s for p, s in brats_patients.items() if len(s) >= 3}
    print(f'\nBraTS-PTG done ({time.time()-t0:.0f}s)')
    print(f'  Patients:      {len(brats_patients)}')
    print(f'  Total scans:   {sum(len(v) for v in brats_patients.values())}')
    print(f'  With ≥3tp:     {len(eval_pats)} (usable for eval)')
    print(f'  Corrupted:     {len(brats_corrupted)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 7: Build & Save BraTS-PTG Evaluation Triplets
# ═══════════════════════════════════════════════════════════════
e_starts, e_ends, e_gts, e_t1cs, e_ts, e_pids = [], [], [], [], [], []
eval_patients = {p: s for p, s in brats_patients.items() if len(s) >= 3}

for pid, scans in eval_patients.items():
    for i in range(len(scans)):
        for k in range(i+2, len(scans)):
            for j in range(i+1, k):
                t_val = (j - i) / (k - i)
                e_starts.append(scans[i]['seg']); e_ends.append(scans[k]['seg'])
                e_gts.append(scans[j]['seg']); e_t1cs.append(scans[i]['t1c'])
                e_ts.append(t_val); e_pids.append(pid)

print(f'BraTS-PTG eval triplets: {len(e_starts)}')
if e_starts:
    save_path = OUTPUT_DIR / 'brats_ptg_eval_triplets.npz'
    np.savez_compressed(str(save_path),
        seg_start=np.array(e_starts, dtype=np.uint8),
        seg_end=np.array(e_ends, dtype=np.uint8),
        seg_gt=np.array(e_gts, dtype=np.uint8),
        t1c_start=np.array(e_t1cs, dtype=np.float32),
        t_interp=np.array(e_ts, dtype=np.float32),
        pids=np.array(e_pids))
    print(f'✅ Saved → {save_path}  ({save_path.stat().st_size/1e6:.1f} MB)')
else:
    print('  ⚠ No triplets (need patients with ≥3 timepoints)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 8: Visualize Sample Slices
# ═══════════════════════════════════════════════════════════════
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

def label_to_rgb(seg):
    rgb = np.zeros((*seg.shape, 3), dtype=np.float32)
    rgb[seg == 1] = [0.2, 0.4, 1.0]   # NCR - blue
    rgb[seg == 2] = [0.2, 0.8, 0.3]   # ED  - green
    rgb[seg == 4] = [1.0, 0.2, 0.2]   # ET  - red
    return rgb

example_pids = list(mu_patients.keys())[:4]
n_pats = len(example_pids)
fig, axes = plt.subplots(n_pats, 6, figsize=(18, 3*n_pats))
fig.suptitle('MU-Glioma — Sample Patient Slices (T1c + Seg Overlay)', fontsize=14, fontweight='bold')

for row, pid in enumerate(example_pids):
    scans = mu_patients[pid]
    for col in range(6):
        ax = axes[row, col] if n_pats > 1 else axes[col]
        if col < len(scans):
            s = scans[col]
            t1c_rgb = np.stack([s['t1c']]*3, axis=-1)
            seg_rgb = label_to_rgb(s['seg'])
            overlay = np.where(seg_rgb.sum(-1, keepdims=True) > 0,
                              0.4*t1c_rgb + 0.6*seg_rgb, t1c_rgb)
            ax.imshow(overlay, origin='lower')
            ax.set_title(f'TP {s["tp_num"]}\nWT={s["wt"]}px', fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'sample_slices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sample visualization saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 9: Summary
# ═══════════════════════════════════════════════════════════════
print('\n' + '═'*60)
print('  VIDEO DATA PREPARATION — COMPLETE')
print('═'*60)
print(f'\n  MU-Glioma (Training):')
print(f'    Patients (≥2tp): {len(mu_patients)}')
print(f'    Scans:           {sum(len(v) for v in mu_patients.values())}')
print(f'    Triplets:        {n_triplet} real + {n_identity} identity = {n_triplet+n_identity}')
print(f'\n  BraTS-PTG (Validation):')
if brats_patients:
    print(f'    Patients:        {len(brats_patients)}')
    print(f'    Eval (≥3tp):     {len(eval_patients)} patients, {len(e_starts)} triplets')
else:
    print(f'    Not attached')
print(f'\n  Output files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'    {f.name:45s} {f.stat().st_size/1e6:7.1f} MB')
print(f'\n  → Next: Video_B_Train.ipynb')
print('═'*60)